# Demostración: enfermedades en hojas de café

Clasifica una foto de hoja de café en **sana**, **roya** u **ojo de gallo** con el modelo ViT-B/16 entrenado con fotos de Saposoa (San Martín, Perú), con aumento fuerte y reemplazo de fondo (semilla 0).

Funciona con o sin GPU. Para usarlo: *Entorno de ejecución → Ejecutar todas*.

**Antes de usarlo:** el modelo se entrenó con hojas sueltas sobre fondo claro. En fotos tomadas en campo detecta bien la roya, pero confunde muchas hojas sanas con enfermas y a menudo responde "ojo de gallo". Los detalles están en el artículo y en el README del repositorio.

In [ ]:
!git clone -q https://github.com/pomazon/Examen_Final_Redes_Neuronales.git proyecto
%cd proyecto
!pip install -q gdown

Descarga de los pesos entrenados (Google Drive, público).

In [ ]:
PESOS_URL = "https://drive.google.com/file/d/1EGhP7CiMcwePDhZN2GA95uB0fhsncG5I/view?usp=sharing"
!gdown -q --fuzzy "{PESOS_URL}" -O vit_b_16_fondo_s0.pt

In [ ]:
import sys, glob
import torch
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image

sys.path.insert(0, "src")
from entrenar import crear_modelo
from aumentos import transformacion
from metricas import CLASES

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo = crear_modelo("vit_b_16", preentrenado=False)
modelo.load_state_dict(torch.load("vit_b_16_fondo_s0.pt", map_location=dispositivo))
modelo.eval().to(dispositivo)
t = transformacion("evaluacion")

def predecir(im):
    with torch.no_grad():
        x = t(im.convert("RGB")).unsqueeze(0).to(dispositivo)
        return dict(zip(CLASES, torch.softmax(modelo(x), 1)[0].cpu().tolist()))

## Imágenes de ejemplo

Siete imágenes que el modelo no vio al entrenar: una de cada clase de Saposoa (prueba) y una sana y una con roya de RoCoLe (Ecuador) y de Uganda.

In [ ]:
fuentes = pd.read_csv("data/ejemplos/fuentes.csv")
fig, ejes = plt.subplots(1, len(fuentes), figsize=(2.6 * len(fuentes), 3.4))
for eje, r in zip(ejes, fuentes.itertuples()):
    im = Image.open(f"data/ejemplos/{r.archivo}")
    p = predecir(im)
    eje.imshow(im)
    eje.axis("off")
    texto = "\n".join(f"{c}: {v:.0%}" for c, v in sorted(p.items(), key=lambda x: -x[1]))
    eje.set_title(f"{r.dataset}, real: {r.clase}\n{texto}", fontsize=8)
fig.tight_layout()
plt.show()

## Probar con una foto propia

In [ ]:
from google.colab import files

for nombre in files.upload():
    im = Image.open(nombre)
    p = predecir(im)
    plt.figure(figsize=(3.5, 3.5))
    plt.imshow(im.convert("RGB"))
    plt.axis("off")
    plt.title(f"{max(p, key=p.get)}  ({max(p.values()):.0%})")
    plt.show()
    for c, v in sorted(p.items(), key=lambda x: -x[1]):
        print(f"{c:10s} {v:.1%}")